In [ ]:
from dotenv import load_dotenv

load_dotenv()

The environment variables are now loaded from the local `.env` file. This makes the OpenAI API key available without storing it directly in the notebook.

In [ ]:
from openai import OpenAI

openai = OpenAI()

The OpenAI client will be used to generate a concise summary for each stock. Next, we define the Yahoo Finance starting page and the maximum number of losing stocks to analyze.

In [ ]:
website_url = "https://finance.yahoo.com/"
top_losers_limit = 5

These settings keep the data source and result limit easy to change. The following cell loads the scraping helpers and defines how the content of each ticker page is summarized.

In [ ]:
import importlib
import scraper

importlib.reload(scraper)
from scraper import fetch_website_contents, fetch_yahoo_top_losers


def summarize(ticker: str, ticker_url: str) -> str:
    """Analizza la pagina Yahoo Finance di un ticker e restituisce una breve sintesi."""
    ticker_contents = fetch_website_contents(ticker_url)
    if not ticker_contents.strip():
        raise ValueError(f"La pagina di {ticker} non contiene testo analizzabile")

    completion = openai.chat.completions.create(
        model="gpt-4.1-nano",
        messages=[
            {
                "role": "system",
                "content": (
                    "Sei un analista finanziario sintetico e prudente. Riassumi in italiano i dati presenti "
                    "nella pagina Yahoo Finance del titolo, spiegando società, andamento mostrato e possibili "
                    "elementi rilevanti. Non inventare dati e non fornire raccomandazioni di investimento. "
                    "Produci un unico breve paragrafo, senza titolo e senza tabella."
                ),
            },
            {
                "role": "user",
                "content": f"Ticker: {ticker}\nURL: {ticker_url}\n\nContenuto della pagina:\n{ticker_contents}",
            },
        ],
    )

    return completion.choices[0].message.content

The `summarize` function extracts readable page content and asks the model for a cautious summary in Italian. The final cell retrieves the top losers, summarizes them one at a time, and presents the results in a Markdown table.

In [ ]:
from IPython.display import Markdown, display

top_losers = fetch_yahoo_top_losers(website_url, limit=top_losers_limit)
summaries = []

for item in top_losers:
    ticker_summary = summarize(item["ticker"], item["url"])
    summaries.append({**item, "summary": ticker_summary})

markdown_rows = [
    "# Yahoo Finance — Top Losers",
    "",
    "| Ticker | Sintesi |",
    "|---|---|",
]
for item in summaries:
    clean_summary = item["summary"].replace("\n", " ").replace("|", "\\|")
    markdown_rows.append(f'| [{item["ticker"]}]({item["url"]}) | {clean_summary} |')

final_report = "\n".join(markdown_rows)
display(Markdown(final_report))